# Step 3B — EDA of ALSFRS-R slopes (3m and 6m)

## Objective
Analyse the quality and behaviour of the ALSFRS-R slopes computed in Step 3A, including:
- **Eligibility/coverage** per horizon (3m, 6m, both)
- **Slope quality** (number of data points and presence of measurement near t0+H)
- **Distribution** of slopes (histograms)
- **Intra-patient consistency** between 3m and 6m (scatter 3m vs 6m)

## Input
- `01_data/interim/baseline_with_targets_step3_nolabelsv2.csv`  *(1 row per patient)*

## Outputs (tables/figures)
- `04_outputs/tables/step3_eligibility_summary.csv`
- `04_outputs/tables/step3_slope_stats.csv`
- `04_outputs/tables/step3_slope_quality_points.csv`
- `04_outputs/figures/step3_hist_slope_3m*.png`
- `04_outputs/figures/step3_hist_slope_6m*.png`
- `04_outputs/figures/step3_scatter_3m_vs_6m*.png`

<div style="padding:10px;border-left:6px solid #FF5F5D;">
<b>Note:</b> this notebook is descriptive (EDA). The binary "rapid" definition (top 30%) is used here only for visual reference; the actual threshold decision is applied during training (fold-wise).
</div>


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Project paths ===
DATA_INTERIM = os.path.join("..", "01_data", "interim")
OUT_TABLES   = os.path.join("..", "04_outputs", "tables", "step3")
OUT_FIGURES  = os.path.join("..", "04_outputs", "figures", "step3_slopes")

os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_FIGURES, exist_ok=True)

# === Step 3 file (baseline + slopes) — v2 (BMI correction) ===
FILE = os.path.join(DATA_INTERIM, "baseline_with_targets_step3_nolabelsv2.csv")

df = pd.read_csv(FILE)

print("Rows:", df.shape[0], "| Columns:", df.shape[1])
df.head()


## 1) Read the intermediate dataset and basic validation

In this section we verify:
- columns required for EDA (slopes and quality metrics)
- types and presence of null values
- basic consistency: 1 row per patient


In [ ]:
needed_cols = [
    "subject_id",
    "slope_90d_per_30d",
    "slope_180d_per_30d",
    "npoints_90d_window",
    "npoints_180d_window",
    "has_endband_90d",
    "has_endband_180d"
]

missing = [c for c in needed_cols if c not in df.columns]
print("Missing columns:", missing)

df[needed_cols].describe(include="all")


## 2) Eligibility (coverage): how many patients have a slope at 3m, 6m, and both?

A patient is considered eligible for a horizon if:
- a slope was computed for that horizon (not null)

We report:
- N and % eligible for 3m
- N and % eligible for 6m
- N and % eligible for both (useful for consistency analyses)


In [ ]:
N = len(df)

ok3  = df["slope_90d_per_30d"].notna()
ok6  = df["slope_180d_per_30d"].notna()
both = ok3 & ok6

summary = pd.DataFrame({
    "horizon": ["3m (90±7d)", "6m (180±7d)", "Both (3m & 6m)"],
    "n_eligible": [ok3.sum(), ok6.sum(), both.sum()],
})
summary["pct"] = summary["n_eligible"] / N

summary


## 2.1) Save eligibility summary

We save this table because it is direct evidence of:
- actual PRO-ACT coverage for the chosen horizons
- effective sample size for training and evaluation (Step 5)


In [ ]:
out = os.path.join(OUT_TABLES, "step3_eligibility_summary.csv")
summary.to_csv(out, index=False)
print("Saved:", out)


## 3) Descriptive statistics of slopes (percentiles and spread)

Here we describe the overall behaviour of the slope (points/month):
- central tendency (median)
- spread (IQR/percentiles)
- extremes (min/max) — useful for identifying outliers

This helps justify:
- choosing the "top 30%" as the rapid-progression region
- expected differences between 3m and 6m (sample size and variability)


In [ ]:
def slope_stats(series: pd.Series, name: str) -> pd.DataFrame:
    s = series.dropna()
    return pd.DataFrame([{
        "target": name,
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(),
        "p10": s.quantile(0.10),
        "p25": s.quantile(0.25),
        "p50": s.quantile(0.50),
        "p75": s.quantile(0.75),
        "p90": s.quantile(0.90),
        "min": s.min(),
        "max": s.max(),
    }])

stats = pd.concat([
    slope_stats(df["slope_90d_per_30d"],  "slope_3m (pts/30d)"),
    slope_stats(df["slope_180d_per_30d"], "slope_6m (pts/30d)"),
], ignore_index=True)

stats


## 3.1) Save descriptive statistics (for thesis reporting)

This table is thesis-ready and can be used directly in the Results section:
- one row per horizon
- quick comparison between 3m and 6m


In [ ]:
out = os.path.join(OUT_TABLES, "step3_slope_stats.csv")
stats.to_csv(out, index=False)
print("Saved:", out)


## 4) Histograms of slopes (3m and 6m)

Histogram objectives:
- visualise the slope distribution (decline vs stability vs improvement)
- compare the shape of the distribution between 3m and 6m
- highlight visual references:
  - line at 0 (stable)
  - cut-off (top 30% worst slopes) as exploratory indication

<b>Note on bins:</b> "bins" are the intervals on the X-axis that group slope values.
More bins = more detail (but more noise); fewer bins = more smoothing (but less detail).


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# --- 1) compute common scale (1% to 99% across both horizons) ---
s3 = df["slope_90d_per_30d"].dropna()
s6 = df["slope_180d_per_30d"].dropna()

x_min_common = float(min(s3.quantile(0.01), s6.quantile(0.01)))
x_max_common = float(max(s3.quantile(0.99), s6.quantile(0.99)))

print("Common X scale:", x_min_common, "to", x_max_common)

# --- 2) plotting function with fixed limits ---
def plot_hist_slope_common(df, col, horizon_label, outname, bins=35, shade_rapid=True,
                           x_min=None, x_max=None):
    s = df[col].dropna()

    # clip to keep the same scale and prevent outliers from distorting
    s_plot = s.clip(x_min, x_max)

    cutoff = float(s.quantile(0.30))  # EDA: 30% most negative

    plt.figure(figsize=(10, 5))

    plt.hist(
        s_plot,
        bins=bins,
        density=True,
        edgecolor="white",
        linewidth=1.0,
        alpha=0.75
    )

    if shade_rapid:
        plt.axvspan(x_min, min(cutoff, x_max), color="red", alpha=0.08, label="Rapid zone (EDA)")

    plt.axvline(0, color="black", linewidth=2, linestyle="--", label="0 (stable)")
    plt.axvline(cutoff, color="red", linewidth=2.5, linestyle="-", label="Cut-off top 30% (EDA)")

    plt.title(f"ALSFRS-R Slope ({horizon_label}) | N={len(s)}")
    plt.xlabel("Slope (points/month)")
    plt.ylabel("Density")

    plt.xlim(x_min, x_max)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out = os.path.join(OUT_FIGURES, outname)
    plt.savefig(out, dpi=300)
    plt.show()
    print("Saved:", out)

# --- 3) generate the 2 histograms with the same scale ---
plot_hist_slope_common(
    df, "slope_90d_per_30d",
    "3 months (90±7 days)",
    "step3_hist_slope_3m_commonX.png",
    bins=35, shade_rapid=True,
    x_min=x_min_common, x_max=x_max_common
)

plot_hist_slope_common(
    df, "slope_180d_per_30d",
    "6 months (180±7 days)",
    "step3_hist_slope_6m_commonX.png",
    bins=35, shade_rapid=True,
    x_min=x_min_common, x_max=x_max_common
)


## 5) Consistency 3m vs 6m (patients with both slopes)

The scatter plot (3m vs 6m) helps to understand:
- whether patients who decline early also tend to decline later
- whether there are "mixed" cases (e.g., worsens at 3m but stabilises at 6m, or vice-versa)

Quick interpretation by quadrant:
- x<0 and y<0: decline in both horizons
- x>0 and y>0: improvement in both (less frequent)
- mixed: non-monotonic patterns / variability / measurement noise


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# assumes you already have:
# df, OUT_FIGURES

both = df["slope_90d_per_30d"].notna() & df["slope_180d_per_30d"].notna()
x = df.loc[both, "slope_90d_per_30d"].to_numpy()
y = df.loc[both, "slope_180d_per_30d"].to_numpy()

# --- correlations (quantify what the eye sees) ---
rp = pearsonr(x, y)[0]
rs = spearmanr(x, y)[0]

# --- masks by quadrant / group ---
decline_both = (x < 0) & (y < 0)
improve_both = (x > 0) & (y > 0)

# mixed in 2 subgroups (direction)
mixed_3m_decline_6m_improve = (x < 0) & (y > 0)   # improves at 6m
mixed_3m_improve_6m_decline = (x > 0) & (y < 0)   # worsens at 6m

# cases exactly at 0 (or close to 0) — kept as "mixed"
mixed_zero = ~(decline_both | improve_both |
               mixed_3m_decline_6m_improve | mixed_3m_improve_6m_decline)

# --- equal limits (1% to 99%) to avoid flattening by outliers ---
lim_lo, lim_hi = np.quantile(np.concatenate([x, y]), [0.01, 0.99])
lim_lo, lim_hi = float(lim_lo), float(lim_hi)

plt.figure(figsize=(7.5, 7.5))

# points (coloured by group)
plt.scatter(x[decline_both], y[decline_both],
            s=14, alpha=0.45, color="red",
            label=f"Decline in both (N={decline_both.sum()})")

plt.scatter(x[improve_both], y[improve_both],
            s=14, alpha=0.55, color="green",
            label=f"Improvement in both (N={improve_both.sum()})")

plt.scatter(x[mixed_3m_decline_6m_improve], y[mixed_3m_decline_6m_improve],
            s=14, alpha=0.40, color="deepskyblue",
            label=f"Mixed: 3m<0 & 6m>0 (N={mixed_3m_decline_6m_improve.sum()})")

plt.scatter(x[mixed_3m_improve_6m_decline], y[mixed_3m_improve_6m_decline],
            s=14, alpha=0.40, color="navy",
            label=f"Mixed: 3m>0 & 6m<0 (N={mixed_3m_improve_6m_decline.sum()})")

if mixed_zero.any():
    plt.scatter(x[mixed_zero], y[mixed_zero],
                s=14, alpha=0.35, color="gray",
                label=f"Zeros/boundary (N={mixed_zero.sum()})")

# reference lines (lighter)
plt.axhline(0, linewidth=1.2, alpha=0.65)
plt.axvline(0, linewidth=1.2, alpha=0.65)

# y=x line (perfect consistency)
plt.plot([lim_lo, lim_hi], [lim_lo, lim_hi],
         linestyle="--", linewidth=2, label="y = x")

# title + correlation
plt.title(f"ALSFRS-R slopes at 3 vs 6 months (N={len(x)})\n"
          f"Pearson={rp:.2f} | Spearman={rs:.2f}")
plt.xlabel("3 months (points/month)")
plt.ylabel("6 months (points/month)")

plt.xlim(lim_lo, lim_hi)
plt.ylim(lim_lo, lim_hi)

plt.grid(True, alpha=0.25)
plt.legend(loc="upper left", frameon=True)
plt.tight_layout()

out = os.path.join(OUT_FIGURES, "step3_scatter_3m_vs_6m_quadrants_v2.png")
plt.savefig(out, dpi=300)
plt.show()

print("Saved:", out)


## 4) Slope quality (robustness of the computation)

Before using the slopes as a basis for defining "rapid vs slow", we validate the quality of the computation.  
The goal here is not to "improve" the slopes, but to **characterise** their robustness.

### What we check
- **Number of points used in the fit** (e.g., 2, 3, 4+ measurements)
- **Effective temporal window** (how many days between the first and last measurement used)
- **Presence of measurement near the target** (whether a data point exists close to t0+H within ±7 days)

### Why this matters
- With **2 points**, the slope is essentially a difference between baseline and follow-up (more sensitive to noise).
- With **3+ points**, the linear regression fit tends to be more stable.
- If there is no measurement near t0+H, the slope may be less representative of the horizon (even if it mathematically "exists").

This section justifies, in the thesis, that the targets were built with minimal quality control and that the limitations (coverage/measurements) are known and reported.


In [ ]:
qual = pd.DataFrame({
    "horizon": ["3m", "6m"],
    "pct_npoints_ge3": [
        (df.loc[ok3, "npoints_90d_window"] >= 3).mean(),
        (df.loc[ok6, "npoints_180d_window"] >= 3).mean(),
    ],
    "median_npoints": [
        df.loc[ok3, "npoints_90d_window"].median(),
        df.loc[ok6, "npoints_180d_window"].median(),
    ]
})

qual


## 4.1) Save slope quality summary

We save a thesis-ready summary of slope quality per horizon, including:
- counts by number of points (e.g., 2 vs ≥3)
- percentages (for comparing 3m vs 6m)
- (optional) temporal span statistics

This is useful for:
- justifying methodological choices
- explaining differences between horizons (e.g., 6m having fewer patients and sometimes fewer measurements)
- supporting the limitations / Threats to Validity section


In [ ]:
out = os.path.join(OUT_TABLES, "step3_slope_quality_npoints.csv")
qual.to_csv(out, index=False)
print("Saved:", out)
